In [15]:
import pandas as pd
import boto3
import json
import os
import gzip
import datetime
import pytz

In [16]:
s3 = boto3.client('s3')
bucket_name = 'library-ai-ers'

In [17]:
# Get list of all CSV files in data folder
data_files = []
data_folder = 'data'  # Go up one directory level from notebooks/ to find data/
for file in os.listdir(data_folder):
    if file.endswith('.csv'):
        data_files.append(os.path.join(data_folder, file))

# Read and concatenate all CSV files
dfs = []
for file in data_files:
    df = pd.read_csv(file)
    dfs.append(df)

# Combine all dataframes
if dfs:
    df = pd.concat(dfs, ignore_index=True)
    
    # Convert timestamp column to datetime
    #df['originalTimestamp'] = pd.to_datetime(df['originalTimestamp'])
    
    # Get the latest date
    latest_date = df['originalTimestamp'].max()
    print(f"Latest date in data: {latest_date}")
else:
    latest_date = datetime.datetime.now().strftime('%Y-%m-%d')
    print("No CSV files found in data folder")


No CSV files found in data folder


In [18]:
s3_folder = '/segment-logs/o3zpZamYbbVCZCuRCnu1MX/'
files = []
file_dates = []
paginator = s3.get_paginator('list_objects_v2')
pages = paginator.paginate(Bucket=bucket_name, Prefix=s3_folder.lstrip('/'))

for page in pages:
    for obj in page.get('Contents', []):
        files.append(obj['Key'])
        file_dates.append(obj['LastModified'])

# Create a dictionary mapping filenames to their upload dates
file_info = dict(zip(files, file_dates))

In [19]:
def download_import_data(file_name, file_date):
    # Skip if file date is not newer than latest_date
    latest_date_formatted = datetime.datetime.strptime(latest_date.split()[0], '%Y-%m-%d')
    latest_date_formatted = latest_date_formatted.replace(tzinfo=pytz.UTC)
    
    if file_date <= latest_date_formatted:
        return pd.DataFrame()
        
    local_file_name = os.path.basename(file_name)
    with open(local_file_name, 'wb') as f:
        s3.download_fileobj(bucket_name, file_name, f)

    # Read the gzipped JSON file into a DataFrame
    # Segment typically stores data as newline-delimited JSON (NDJSON)
    df = pd.DataFrame()

    with gzip.open(local_file_name, 'rt') as f:
        # Read file line by line since each line is a separate JSON object
        data = [json.loads(line) for line in f]
        df = pd.DataFrame(data)

    # Expand the context column into separate columns
    context_df = pd.json_normalize(df['context'])
    # Drop the original context column and join with expanded columns
    df = df.drop('context', axis=1).join(context_df, rsuffix='_context')
    try:
        properties_df = pd.json_normalize(df['properties'])
        df = df.drop('properties', axis=1).join(properties_df, rsuffix='_properties')
    except:
        pass

    os.remove(local_file_name)

    return df

In [20]:
full_df = pd.DataFrame()

for file, file_date in file_info.items():
    df = download_import_data(file, file_date)
    full_df = pd.concat([full_df, df])

print(full_df.shape)
print(full_df.columns)
full_df.head()

(387, 21)
Index(['channel', 'anonymousId', 'type', 'event', 'messageId',
       '__segment_internal', 'timestamp', 'userId', 'sentAt', 'receivedAt',
       'projectId', 'version', 'integrations', 'originalTimestamp', 'name',
       'category', 'library.name', 'library.version', 'user_input',
       'search_result', 'assistant_response'],
      dtype='object')


,channel,anonymousId,type,event,messageId,__segment_internal,timestamp,userId,sentAt,receivedAt,...,version,integrations,originalTimestamp,name,category,library.name,library.version,user_input,search_result,assistant_response
0,server,None,track,user_input,6e34f3c8-c72a-4f8a-ad67-335fd9df6d97,{},2025-08-13T23:02:25.113Z,null,2025-08-13T23:02:25.058Z,2025-08-13T23:02:25.135Z,...,2,{},2025-08-13T23:02:25.037+00:00,NaN,NaN,analytics-python,2.3.4,lock ness monster,NaN,NaN
1,server,None,track,search_result,fa6124b6-770e-479a-8f4b-789994985533,{},2025-08-13T23:54:26.676Z,null,2025-08-13T23:54:26.901Z,2025-08-13T23:54:26.910Z,...,2,{},2025-08-13T23:54:26.668+00:00,NaN,NaN,analytics-python,2.3.4,NaN,"{""_id"": ""9781426339233"", ""_score"": 0.127001553...",NaN
2,server,None,page,NaN,26d30df7-3bdb-4000-abe3-081cd8aad6ec,{},2025-08-13T23:56:33.672Z,null,2025-08-13T23:56:33.674Z,2025-08-13T23:56:33.684Z,...,2,{},2025-08-13T23:56:33.663+00:00,whitman-chat,NaN,analytics-python,2.3.4,NaN,NaN,NaN
0,server,None,track,search_result,e0940e9e-3bf1-4b12-ab33-3812bed48daa,{},2025-08-13T23:35:13.390Z,null,2025-08-13T23:35:13.745Z,2025-08-13T23:35:13.824Z,...,2,{},2025-08-13T23:35:13.312+00:00,NaN,NaN,analytics-python,2.3.4,NaN,"{""_id"": ""9780545523042"", ""_score"": 0.001995538...",NaN
0,server,None,track,search_result,fd43f17d-b57d-45d7-a89e-b5c7a02e10d0,{},2025-08-13T23:35:13.389Z,null,2025-08-13T23:35:13.745Z,2025-08-13T23:35:13.824Z,...,2,{},2025-08-13T23:35:13.311+00:00,NaN,NaN,analytics-python,2.3.4,NaN,"{""_id"": ""9780545840750"", ""_score"": 0.004591683...",NaN
